# Social Post Strategy Analysis\n\n## 1. Problem Framing\n\nLeadership relies on social media to reach potential donors, but they do not yet know which post characteristics are most associated with stronger donation results. This notebook uses an **explanatory** pipeline to estimate how platform, content theme, CTA style, timing, and basic post structure relate to short-term donation revenue.\n\nThe intended business use is **decision support for the social media chatbot**: the chatbot should keep generating posts, but it should do so with evidence-backed guidance about which post characteristics appear to align with stronger revenue outcomes.\n\nThis notebook is explanatory rather than predictive. The goal is not to forecast exact revenue for new posts; it is to understand which characteristics are most strongly associated with donation success and to package those findings for the web app.\n

In [ ]:
from __future__ import annotations\n\nimport json\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport statsmodels.formula.api as smf\n\nBASE = Path.cwd()\nCSV_PATH = BASE / 'data' / 'social_post_performance.csv'\nOUTPUT_PATH = BASE / 'social_post_strategy_metadata.json'\n

## 2. Data Acquisition, Preparation & Exploration\n\nThe preferred input is a post-level export at `ml-pipelines/data/social_post_performance.csv` with one row per published post and an attributable 7-day revenue outcome.\n\nExpected columns:\n- `platform`\n- `content_theme`\n- `cta_type`\n- `time_bucket`\n- `day_of_week`\n- `content_type`\n- `caption_length`\n- `hashtag_count`\n- `used_image`\n- `revenue_7d_php`\n- `donations_7d`\n\nIf that export is not available yet, the notebook falls back to a tiny demo dataset so the notebook remains executable. Demo-mode results should **not** be used operationally.\n

In [ ]:
demo_rows = [\n    {'platform': 'Facebook', 'content_theme': 'ImpactStory', 'cta_type': 'DonateNow', 'time_bucket': 'Evening', 'day_of_week': 'Tuesday', 'content_type': 'Post', 'caption_length': 122, 'hashtag_count': 3, 'used_image': 1, 'revenue_7d_php': 6800, 'donations_7d': 6},\n    {'platform': 'Facebook', 'content_theme': 'ImpactStory', 'cta_type': 'DonateNow', 'time_bucket': 'Evening', 'day_of_week': 'Thursday', 'content_type': 'Post', 'caption_length': 140, 'hashtag_count': 2, 'used_image': 1, 'revenue_7d_php': 7200, 'donations_7d': 7},\n    {'platform': 'Facebook', 'content_theme': 'UrgentNeed', 'cta_type': 'GiveToday', 'time_bucket': 'Evening', 'day_of_week': 'Saturday', 'content_type': 'Post', 'caption_length': 118, 'hashtag_count': 2, 'used_image': 1, 'revenue_7d_php': 6100, 'donations_7d': 5},\n    {'platform': 'Facebook', 'content_theme': 'TrustBuilding', 'cta_type': 'LearnMore', 'time_bucket': 'Afternoon', 'day_of_week': 'Wednesday', 'content_type': 'Post', 'caption_length': 165, 'hashtag_count': 1, 'used_image': 1, 'revenue_7d_php': 3300, 'donations_7d': 3},\n    {'platform': 'Facebook', 'content_theme': 'ImpactStory', 'cta_type': 'DonateNow', 'time_bucket': 'Morning', 'day_of_week': 'Monday', 'content_type': 'Video', 'caption_length': 98, 'hashtag_count': 4, 'used_image': 1, 'revenue_7d_php': 4200, 'donations_7d': 4},\n    {'platform': 'Facebook', 'content_theme': 'Education', 'cta_type': 'LearnMore', 'time_bucket': 'Afternoon', 'day_of_week': 'Tuesday', 'content_type': 'Carousel', 'caption_length': 180, 'hashtag_count': 5, 'used_image': 1, 'revenue_7d_php': 2500, 'donations_7d': 2},\n    {'platform': 'Instagram', 'content_theme': 'ImpactStory', 'cta_type': 'DonateNow', 'time_bucket': 'Evening', 'day_of_week': 'Thursday', 'content_type': 'Story', 'caption_length': 70, 'hashtag_count': 6, 'used_image': 1, 'revenue_7d_php': 2100, 'donations_7d': 2},\n    {'platform': 'Instagram', 'content_theme': 'UrgentNeed', 'cta_type': 'GiveToday', 'time_bucket': 'Evening', 'day_of_week': 'Friday', 'content_type': 'Story', 'caption_length': 64, 'hashtag_count': 7, 'used_image': 1, 'revenue_7d_php': 2600, 'donations_7d': 2},\n    {'platform': 'Instagram', 'content_theme': 'TrustBuilding', 'cta_type': 'LearnMore', 'time_bucket': 'Morning', 'day_of_week': 'Monday', 'content_type': 'Post', 'caption_length': 102, 'hashtag_count': 8, 'used_image': 1, 'revenue_7d_php': 1400, 'donations_7d': 1},\n    {'platform': 'Instagram', 'content_theme': 'Education', 'cta_type': 'LearnMore', 'time_bucket': 'Afternoon', 'day_of_week': 'Wednesday', 'content_type': 'Carousel', 'caption_length': 150, 'hashtag_count': 8, 'used_image': 1, 'revenue_7d_php': 1200, 'donations_7d': 1},\n    {'platform': 'Instagram', 'content_theme': 'ReintegrationWin', 'cta_type': 'DonateNow', 'time_bucket': 'Evening', 'day_of_week': 'Sunday', 'content_type': 'Video', 'caption_length': 88, 'hashtag_count': 5, 'used_image': 1, 'revenue_7d_php': 2800, 'donations_7d': 3},\n    {'platform': 'Facebook', 'content_theme': 'ReintegrationWin', 'cta_type': 'DonateNow', 'time_bucket': 'Evening', 'day_of_week': 'Sunday', 'content_type': 'Video', 'caption_length': 105, 'hashtag_count': 3, 'used_image': 1, 'revenue_7d_php': 5400, 'donations_7d': 5}\n]\n\ndemo_mode = not CSV_PATH.exists()\nif demo_mode:\n    df = pd.DataFrame(demo_rows)\nelse:\n    df = pd.read_csv(CSV_PATH)\n\nrequired_columns = {\n    'platform', 'content_theme', 'cta_type', 'time_bucket', 'day_of_week', 'content_type',\n    'caption_length', 'hashtag_count', 'used_image', 'revenue_7d_php', 'donations_7d'\n}\nmissing = sorted(required_columns - set(df.columns))\nif missing:\n    raise ValueError(f'Missing required columns: {missing}')\n\ndf = df.copy()\ndf['revenue_7d_php'] = df['revenue_7d_php'].fillna(0).clip(lower=0)\ndf['donations_7d'] = df['donations_7d'].fillna(0).clip(lower=0)\ndf['caption_length'] = df['caption_length'].fillna(0).clip(lower=0)\ndf['hashtag_count'] = df['hashtag_count'].fillna(0).clip(lower=0)\ndf['used_image'] = df['used_image'].fillna(0).astype(int)\ndf['log_revenue'] = np.log1p(df['revenue_7d_php'])\n\nprint('Demo mode:' if demo_mode else 'Production data:', demo_mode)\nprint('Observations:', len(df))\ndisplay(df.head())\ndisplay(df.groupby('platform')['revenue_7d_php'].agg(['count', 'mean', 'sum']).round(2))\ndisplay(df.groupby('content_theme')['revenue_7d_php'].agg(['count', 'mean', 'sum']).round(2).sort_values('mean', ascending=False))\n

## 3. Modeling & Feature Selection\n\nWe fit an interpretable OLS model on `log(1 + revenue_7d_php)` using post characteristics that would be known at publish time. This is not a randomized experiment, so coefficients should be treated as **associations** rather than proof of causation. Still, the direction and magnitude of these relationships can guide practical content decisions.\n

In [ ]:
formula = (\n    'log_revenue ~ C(platform) + C(content_theme) + C(cta_type) + C(time_bucket) '\n    '+ C(day_of_week) + C(content_type) + caption_length + hashtag_count + used_image'\n)\n\nmodel = smf.ols(formula=formula, data=df).fit()\nprint(model.summary())\n

## 4. Evaluation & Interpretation\n\nBecause this is an explanatory pipeline, the main outputs are coefficient directions, p-values, and practical implications. False confidence is the key risk here: if we overstate weak findings, the founders may shape their content around noise. If we understate useful signals, they may keep posting sporadically without learning.\n

In [ ]:
coef_table = (\n    pd.DataFrame({\n        'feature': model.params.index,\n        'coefficient': model.params.values,\n        'p_value': model.pvalues.values,\n    })\n    .query("feature != 'Intercept'")\n    .assign(abs_coef=lambda x: x['coefficient'].abs())\n    .sort_values(['p_value', 'abs_coef'], ascending=[True, False])\n)\n\ndisplay(coef_table.head(12).round(4))\nprint('R-squared:', round(float(model.rsquared), 4))\n

## 5. Causal and Relationship Analysis (Required)\n\nThis notebook is honest about its limits: unless the organization starts tagging individual posts with attributable donation links, these relationships remain observational. Even with attribution, unobserved confounders such as campaign quality, external events, and audience size may still affect outcomes.\n\nWhat the model can still reveal is which post features appear consistently aligned with stronger or weaker revenue outcomes, which is enough to improve the chatbot prompt and the team's posting habits.\n

In [ ]:
def label_feature(name: str, coef: float) -> str:\n    direction = 'higher' if coef > 0 else 'lower'\n    cleaned = name.replace('C(', '').replace(')', '').replace('[T.', ' = ').replace(']', '')\n    return f'{cleaned} is associated with {direction} 7-day donation revenue.'\n\nvalidated = [\n    label_feature(row.feature, row.coefficient)\n    for row in coef_table.itertuples()\n    if row.p_value < 0.05\n]\n\ndirectional = [\n    label_feature(row.feature, row.coefficient)\n    for row in coef_table.itertuples()\n    if 0.05 <= row.p_value < 0.20\n]\n\ntop_positive = coef_table.sort_values('coefficient', ascending=False).head(3)\nrecommendations = []\nfor row in top_positive.itertuples():\n    recommendations.append({\n        'title': f'Test more of {row.feature}',\n        'detail': label_feature(row.feature, row.coefficient)\n    })\n\ndata_gaps = []\nif demo_mode:\n    data_gaps.append('Notebook is running on the embedded demo dataset. Replace it with an attributed post export before using findings operationally.')\nif 'actual_publish_time' not in df.columns:\n    data_gaps.append('Actual publish timestamps were not provided, so time effects rely on coarse buckets instead of exact posting windows.')\nif 'tracked_link_id' not in df.columns:\n    data_gaps.append('Tracked donation-link identifiers were not provided, so attribution quality may be limited.')\n\nresult = {\n    'pipeline_version': '0.1.0',\n    'last_run': datetime.now(timezone.utc).date().isoformat() if not demo_mode else None,\n    'analysis_mode': 'explanatory',\n    'unit_of_analysis': 'published social media post',\n    'outcome_metric': 'Attributed donation revenue within 7 days (PHP)',\n    'evidence_strength': 'Demo-only scaffold output.' if demo_mode else 'Live post-level explanatory analysis from notebook output.',\n    'n_observations': int(len(df)),\n    'r_squared': round(float(model.rsquared), 4),\n    'validated_findings': validated[:5],\n    'directional_findings': directional[:5],\n    'recommendations': recommendations[:3],\n    'data_gaps': data_gaps,\n}\n\nprint(json.dumps(result, indent=2))\nif demo_mode:\n    print('Demo mode is active, so social_post_strategy_metadata.json was not overwritten.')\nelse:\n    OUTPUT_PATH.write_text(json.dumps(result, indent=2))\n    print(f'Wrote metadata to {OUTPUT_PATH}')\n

## 6. Deployment Notes\n\nWhen the notebook writes `social_post_strategy_metadata.json`, the FastAPI service in [app.py](./app.py) serves it at `/marketing/post-strategy-analysis`. The ASP.NET backend then injects the findings into the social chatbot context so generated posts can reflect the strongest available post-level evidence without replacing the existing chatbot workflow.\n